# SYNAPSE — Knowledge Preparation Pipeline (Colab)

This notebook is a **preparation / evaluation pipeline only** — it does NOT run the live website, and it does **NOT** contain any API keys.

The production website queries its own backend at runtime; this notebook is for:

1. loading approved, organization-licensed wellbeing knowledge sources
2. chunking them into retrieval-sized passages
3. computing embeddings
4. building a vector index
5. evaluating retrieval quality
6. exporting the index so the backend can use it

## Source policy

We only use sources that are:
- explicitly licensed for re-use, OR
- in the public domain, OR
- short permitted excerpts from books / papers the organization has rights to, OR
- internal notes / playbooks / hand-outs the organization has authored.

We do **NOT** scrape or ingest pirated copyrighted books, and we do **NOT** reproduce long verbatim excerpts.

Use this notebook to:
- verify the chunking + embedding strategy is working,
- run a small retrieval evaluation set,
- export the index for the backend to load.

## 1. Environment

Install only the packages we need. Use environment variables for any cloud credentials — never paste keys into the notebook.

In [ ]:
# @title Install dependencies
!pip install -q sentence-transformers faiss-cpu numpy pandas tqdm

In [ ]:
# @title Imports & config
import os
import json
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# The embedding model is small, fast, and free. Override via env var if you want.
EMBED_MODEL = os.environ.get("SYNAPSE_EMBED_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
CHUNK_SIZE = int(os.environ.get("SYNAPSE_CHUNK_SIZE", "450"))   # chars per chunk
CHUNK_OVERLAP = int(os.environ.get("SYNAPSE_CHUNK_OVERLAP", "80"))
RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)

print("EMBED_MODEL:", EMBED_MODEL)
print("CHUNK_SIZE:", CHUNK_SIZE, "CHUNK_OVERLAP:", CHUNK_OVERLAP)

## 2. Approved knowledge sources

Replace the sample source list with your organization-approved materials. The loader returns a list of `{source_id, title, text}` records.

In [ ]:
def load_approved_sources():
    """Return a list of approved knowledge records.
    
    Replace the body of this function with the loader for your organization's
    approved wellbeing knowledge (PDFs, internal notes, licensed corpora).
    Do NOT include pirated or unlicensed copyrighted material.
    """
    # ---- Example seed content (short, original snippets) -------------------
    samples = [
        {
            "source_id": "playbook-sleep",
            "title": "Sleep Playbook",
            "text": (
                "A consistent wind-down routine is one of the strongest predictors of better sleep. "
                "Dim screens 60 minutes before bed, keep the room cool, and do a 5-minute breathing "
                "or body scan. Avoid caffeine after 2pm. If you can't fall asleep within 20 minutes, "
                "get up and do something quiet until you feel sleepy again."
            ),
        },
        {
            "source_id": "playbook-stress",
            "title": "Operational Stress Playbook",
            "text": (
                "When stress spikes, the fastest reset is a long slow exhale. Try inhale 3, hold 2, "
                "exhale 5 for three rounds. After that, name the one thing in your control and the "
                "smallest physical next action. If the load is sustained, talk to a peer or a counsellor "
                "— sustained stress is a signal, not a weakness."
            ),
        },
        {
            "source_id": "playbook-connection",
            "title": "Connection Playbook",
            "text": (
                "Loneliness shrinks our world. Small actions break it: one short message to someone "
                "you trust, ten minutes in a shared space, or a brief phone call. Connection is a "
                "protective factor for both mood and physical health."
            ),
        },
        {
            "source_id": "playbook-when-to-escalate",
            "title": "When to Escalate",
            "text": (
                "If you are in immediate danger, contact local emergency services. If you are having "
                "thoughts of self-harm that are persistent or escalating, reach out to a crisis line "
                "or a trusted person. The companion is supportive but is not a substitute for "
                "qualified professional help."
            ),
        },
    ]
    return samples

sources = load_approved_sources()
print(f"Loaded {len(sources)} approved source(s).")
for s in sources:
    print("  -", s["source_id"], "·", s["title"], "·", len(s["text"]), "chars")

## 3. Chunking

We split each source into overlapping character chunks. Overlap helps the retriever keep context across chunk boundaries.

In [ ]:
def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = text.strip()
    if not text:
        return []
    out, start = [], 0
    while start < len(text):
        end = min(start + size, len(text))
        # try to break on a sentence boundary if we're not at the end
        if end < len(text):
            for sep in (". ", "\n", "; "):
                idx = text.rfind(sep, start, end)
                if idx > start + size // 2:
                    end = idx + len(sep)
                    break
        out.append(text[start:end].strip())
        if end == len(text):
            break
        start = max(end - overlap, start + 1)
    return [c for c in out if c]

def build_chunks(sources):
    chunks = []
    for s in sources:
        parts = chunk_text(s["text"])
        for i, p in enumerate(parts):
            chunks.append({
                "chunk_id": hashlib.sha1(f'{s["source_id"]}::{i}::{p[:40]}'.encode()).hexdigest()[:16],
                "source_id": s["source_id"],
                "title": s["title"],
                "ord": i,
                "text": p,
            })
    return chunks

chunks = build_chunks(sources)
print(f"Built {len(chunks)} chunk(s).")
for c in chunks[:3]:
    print("  -", c["chunk_id"], c["source_id"], "·", len(c["text"]), "chars")

## 4. Embeddings

We embed each chunk with a small, free sentence-transformers model. This is the same shape the backend uses at runtime.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBED_MODEL)
texts = [c["text"] for c in chunks]
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True, convert_to_numpy=True)
print("Embeddings shape:", embeddings.shape)

## 5. Vector index

We build a small FAISS index over the chunk embeddings and run a few sanity queries.

In [ ]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # inner product == cosine when normalized
index.add(embeddings.astype("float32"))
print("Index size:", index.ntotal)

def search(query, k=3):
    q = model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    D, I = index.search(q, k)
    return [(chunks[i], float(D[0][j])) for j, i in enumerate(I[0])]

for q in ["how do I fall asleep faster", "I'm overwhelmed at work", "I feel alone", "I'm thinking about hurting myself"]:
    print("\nQ:", q)
    for hit, score in search(q, k=2):
        print(f"  {score:.3f}  [{hit['source_id']}] {hit['text'][:90]}...")

## 6. Tiny retrieval evaluation

Below is a minimal eval set. Expand with your own labelled queries once you have real data.

In [ ]:
EVAL = [
    {"q": "how do I sleep better",                          "expect": "playbook-sleep"},
    {"q": "I'm really stressed at work",                   "expect": "playbook-stress"},
    {"q": "I feel disconnected from everyone",             "expect": "playbook-connection"},
    {"q": "I want to hurt myself",                          "expect": "playbook-when-to-escalate"},
    {"q": "caffeine at night ruining my sleep",            "expect": "playbook-sleep"},
    {"q": "I can't focus on anything",                      "expect": "playbook-stress"},
    {"q": "miss my family",                                "expect": "playbook-connection"},
    {"q": "when should I call for help",                   "expect": "playbook-when-to-escalate"},
]

correct = 0
rows = []
for ex in EVAL:
    top1, _ = search(ex["q"], k=1)[0]
    ok = top1["source_id"] == ex["expect"]
    correct += int(ok)
    rows.append({"query": ex["q"], "expected": ex["expect"], "got": top1["source_id"], "ok": ok})

df = pd.DataFrame(rows)
print(f"Top-1 accuracy: {correct}/{len(EVAL)}  =  {correct/len(EVAL):.0%}")
df

## 7. Export the index for the backend

Save the chunks + their embeddings to a JSON file the backend can load. The backend's runtime retriever expects `{chunk_id, source_id, title, text, embedding}` records.

In [ ]:
out = []
for c, e in zip(chunks, embeddings):
    out.append({
        "chunk_id": c["chunk_id"],
        "source_id": c["source_id"],
        "title": c["title"],
        "text": c["text"],
        "embedding": [float(x) for x in e.tolist()],
    })

out_path = "synapse_knowledge_index.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump({"model": EMBED_MODEL, "dim": int(embeddings.shape[1]), "chunks": out}, f)
print("Wrote", out_path, "with", len(out), "chunks.")
from google.colab import files
files.download(out_path)

## 8. Backend integration notes

1. The backend exposes a `GET /api/knowledge/retrieve?q=...&k=3` endpoint that returns the top-k chunks. It loads the JSON index produced above at boot.
2. The runtime `/api/chat` endpoint calls this retriever, then prepends the retrieved passages to the LLM prompt. The LLM is then instructed to ground its answer in those passages and refuse to invent content.
3. If no LLM provider is configured, the server falls back to its built-in companion engine — the same one the demo runs.
4. **Never put API keys in this notebook.** Use environment variables / Colab secrets.
5. The website user never runs this notebook. The website only talks to the backend.